# Hyperparameter Tunnig

This script is using the data pipeline to clean the data.
It will use Hyperopt for hyperparameter tuning and safe the best model via mlflow.

The models will be tested against:
- 1 day
- 1 week
- 2 weeks
- 4 weeks
- 1 quarter
- 2 quarters
- 3 quarters
- 4 quarters

As well as based on data need, this will be evaluated based on CV.

The models to be tuned are:
- SARIMAX
- Tripple Exponential Smoothing
- Prophet
- XG Boost
- Linear Regression
- Random Forest
- LSTM
- Temporal Fusion Transformer (TFT)
- Deep Autoregression Models

# Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import sys
import os

# Add the project root to the python path
sys.path.append(os.path.abspath(".."))
from src.processing import DateFeatureTransformer, TimeSeriesWrangler, LagFeatureTransformer, WindowFeatureTransformer


# Loading Data

In [2]:
# define path
path = "../data/raw/"

In [3]:
# oil data
oil_df = pd.read_csv(path + "oil.csv")

# Initialize the wrangler
wrangler = TimeSeriesWrangler(
    date_col='date', 
    fill_col='dcoilwtico', 
    freq='D', 
    fill_method='ffill'
)

# Run the cleaning logic
oil = wrangler.clean(oil_df)

In [4]:
# timeseries data
timeseries_df = pd.read_csv(path + "timeseries.csv")

# Initialize the wrangler
wrangler = TimeSeriesWrangler(
    date_col='date', 
    fill_col='unit_sales', 
    freq='D', 
    fill_method='zeros'
)

# Run the cleaning logic
timeseries = wrangler.clean(timeseries_df)

/Users/moe/Developer/work-projects/MIST/TimeSeries_April2026/corporacion_favorita_grocery_sales_forecasting/src/processing/wrangler.py:41: FutureWarning: SeriesGroupBy.fillna is deprecated and will be removed in a future version. Use obj.ffill() or obj.bfill() for forward or backward filling instead. If you want to fill with a single value, use Series.fillna instead
  df = resampler[self.fill_col].fillna(0)


# Variables

In [5]:
random_seed = 42

# SARIMAX

In [ ]:
from darts.models import Prophet, ARIMA, ExponentialSmoothing
from hyperopt import hp

# Define your models and search spaces
registry = {
    'Prophet': {
        'class': Prophet,
        'space': {
            'n_changepoints': hp.quniform('n_changepoints', 5, 25, 1),
            'seasonality_mode': hp.choice('sm', ['additive', 'multiplicative'])
        }
    },
    'SARIMAX': {
        'class': ARIMA,
        'space': {
            'p': hp.quniform('p', 0, 3, 1),
            'd': hp.choice('d', [0, 1]),
            'q': hp.quniform('q', 0, 3, 1)
        }
    }
}

# Initialize the orchestrator
optimizer = TimeSeriesOptimizer(experiment_name="Ecuador_Sales_Forecasting")

# Run sequentially (Safe for batching)
for name, config in registry.items():
    optimizer.optimize_and_log(
        model_name=name,
        model_class=config['class'],
        space=config['space'],
        series=my_sales_series,      # Your Darts TimeSeries
        horizon=7,                   # 7-day forecast
        metric=mape,                 # Optimization metric
        exog=my_holiday_series,      # Your Darts exogenous TimeSeries
        max_evals=50                 # Run 50 trials per model
    )

# Tripple Exponential Smooting

# Prophet

# XG Boost

# Linear Regression

# Random Forest

# LSTM

# TFT

# Deep Autoregression Models